# Multiclass Classification using BB-MAS dataset

In [ ]:
import pandas as pd
import glob
import os
import warnings
import numpy as np
import joblib
import time
warnings.filterwarnings("ignore")
os.makedirs("outputs", exist_ok=True)

from sklearn.model_selection   import train_test_split
from sklearn.preprocessing     import StandardScaler, LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model      import LogisticRegression
from sklearn.naive_bayes       import GaussianNB
from sklearn.neighbors         import KNeighborsClassifier
from sklearn.svm               import SVC
from sklearn.ensemble          import (RandomForestClassifier,
                                       ExtraTreesClassifier,
                                       HistGradientBoostingClassifier)
from sklearn.neural_network    import MLPClassifier, MLPRegressor
from sklearn.metrics           import (accuracy_score, precision_score,
                                       recall_score, f1_score,
                                       classification_report,
                                       top_k_accuracy_score,
                                       roc_auc_score, roc_curve)
from xgboost                   import XGBClassifier
from lightgbm                  import LGBMClassifier
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

Defaulting to user installation because normal site-packages is not writeable


DEPRECATION: Loading egg at c:\programdata\anaconda3\lib\site-packages\vboxapi-1.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
path = r"C:\Users\Naresh Kumar\Downloads\Continuous-Authentication-Reinforcement-Learning-and-Behavioural-Biometrics-main\Continuous-Authentication-Reinforcement-Learning-and-Behavioural-Biometrics-main\dataset\full_bbmas_dataset\processed_data"
files = glob.glob(os.path.join(path, "*.csv"))    # Find all .csv files in folder
print(f"Total files found: {len(files)}")
if len(files) == 0:
    raise FileNotFoundError(f"No CSV files found in: {path}")

Total files found: 116


## 1. Column structure + dtype check

In [ ]:
columns_set   = None
mismatch_files = []
for file in files:
    df_head = pd.read_csv(file, nrows=1)
    cols    = list(df_head.columns)
    if columns_set is None:
        columns_set = cols
    elif cols != columns_set:
        mismatch_files.append(file)

if mismatch_files:
    print("Mismatch found in these files:")
    for f in mismatch_files:
        print(f"  {f}")
    raise ValueError("Fix mismatched files before merging.")
else:
    print("All files have identical structure")

All files have identical structure


## 2. Merge with per-file logging + safe user_id extraction

In [ ]:
df_list = []
skipped = []
for file in files:
    try:
        df       = pd.read_csv(file)
        basename = os.path.splitext(os.path.basename(file))[0]
        raw_id   = basename.split("_")[0]
        try:
            user_id = int(raw_id)
        except ValueError:
            print(f"  ⚠ Could not parse user_id from '{basename}', skipping.")
            skipped.append(file)
            continue
        df["user_id"] = user_id
        df_list.append(df)
        print(f"  user_{user_id:03d} → {len(df):>6,} rows loaded")
    except Exception as e:
        print(f"Error reading {file}: {e}")
        skipped.append(file)

  user_100 →  1,575 rows loaded
  user_101 →  1,307 rows loaded
  user_102 →  1,400 rows loaded
  user_103 →  1,712 rows loaded
  user_104 →  1,090 rows loaded
  user_105 →  1,781 rows loaded
  user_106 →  1,574 rows loaded
  user_107 →  1,385 rows loaded
  user_108 →  1,981 rows loaded
  user_109 →  1,859 rows loaded
  user_010 →  1,663 rows loaded
  user_110 →  1,657 rows loaded
  user_111 →  1,575 rows loaded
  user_112 →  1,488 rows loaded
  user_113 →  1,632 rows loaded
  user_114 →  1,548 rows loaded
  user_115 →  1,476 rows loaded
  user_116 →  1,482 rows loaded
  user_117 →  1,368 rows loaded
  user_011 →  1,637 rows loaded
  user_012 →  1,961 rows loaded
  user_013 →  2,486 rows loaded
  user_014 →  1,599 rows loaded
  user_015 →    648 rows loaded
  user_016 →  1,797 rows loaded
  user_018 →  1,738 rows loaded
  user_019 →  1,739 rows loaded
  user_001 →  1,530 rows loaded
  user_020 →  1,446 rows loaded
  user_021 →  2,005 rows loaded
  user_022 →  2,052 rows loaded
  user_0

## 3. Memory-efficient concat

In [ ]:
print("\nMerging all files...")
combined_df            = pd.concat(df_list, ignore_index=True)
combined_df["user_id"] = combined_df["user_id"].astype("int64")

before_dedup = len(combined_df)
combined_df  = combined_df.drop_duplicates().reset_index(drop=True)

print(f"Merging completed")
print(f"Shape          : {combined_df.shape}")
print(f"Unique users   : {combined_df['user_id'].nunique()}")
print(f"Files skipped  : {len(skipped)}")
print(f"Duplicate rows : {before_dedup - len(combined_df)} dropped")
print(f"\nuser_id dtype  : {combined_df['user_id'].dtype}")
print(combined_df['user_id'].value_counts().sort_index().head(10))


Merging all files...
Merging completed
Shape          : (193522, 15)
Unique users   : 116
Files skipped  : 0
Duplicate rows : 0 dropped

user_id dtype  : int64
user_id
1     1530
2     1551
3     1608
4     1538
5     1652
6     1424
7     1900
8     1318
9     1660
10    1663
Name: count, dtype: int64


## 4. Save

In [ ]:
output_path = os.path.join(path, "combined_clean.csv")
combined_df.to_csv(output_path, index=False)
print(f"\nFile saved at: {output_path}")


File saved at: C:\Users\Naresh Kumar\Downloads\Continuous-Authentication-Reinforcement-Learning-and-Behavioural-Biometrics-main\Continuous-Authentication-Reinforcement-Learning-and-Behavioural-Biometrics-main\dataset\full_bbmas_dataset\processed_data\combined_clean.csv


In [ ]:
CSV_PATH      = "combined_clean.csv"
WINDOW_SIZE   = 100 # Each sliding window = 100 consecutive keystrokes
STEP_SIZE     = 25 # Window slides forward 25 keystrokes at a time
CAP_PER_USER  = 250 # Max windows per user — prevents dominant users skewing model
TOP_N_KEYS    = 20
TEST_SIZE     = 0.20
RANDOM_STATE  = 42
CORR_THRESH   = 0.97
MIN_USER_ROWS = 800 # Users with fewer than 800 keystrokes dropped
TIME_COLS     = ["hold_time",  # Duration a key was held down
                 "press_to_press",  # Time between pressing two consecutive keys
                 "release_to_press",  # Time from releasing one key to pressing the next
                 "time_diff"]    # time difference between events
DROP_COLS     = ["key_id", "key_id.1", "delta", "operation", "state",
                 "LR", "pX", "pY", "time_since_beginning"]
VOWELS        = set("aeiouAEIOU")
SPECIAL_KEYS  = {"backspace", "shift", "caps_lock", "ctrl", "alt",
                 "tab", "enter", "escape", "space"}
BANNER        = "=" * 60

## 5. Load Dataset

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"  Shape   : {df.shape}")
print(f"  Missing : {df.isnull().sum().sum()}")
print(f"  Dupes   : {df.duplicated().sum()}")

  Shape   : (193522, 15)
  Missing : 0
  Dupes   : 0


## 6. Drop cols

In [ ]:
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)
before = len(df)
df = df[df["release_to_press"] >= -2.0].copy()  # Remove physically impossible timing rows
print(f" Removed {before - len(df)} rows (release_to_press < -2s)")

 Removed 25 rows (release_to_press < -2s)


## 7. Drop low-data users

In [ ]:
user_counts = df["user_id"].value_counts()
low_users   = user_counts[user_counts < MIN_USER_ROWS].index.tolist()
if low_users:
    print(f" Dropping users {low_users} (< {MIN_USER_ROWS} rows)")
    df = df[~df["user_id"].isin(low_users)].copy()
print(f" Users remaining: {df['user_id'].nunique()}")

 Dropping users [15] (< 800 rows)
 Users remaining: 115


## 8. Per-user timing normalization (by user median)

In [ ]:
for col in TIME_COLS:
    medians  = df.groupby("user_id")[col].transform("median")
    medians  = medians.replace(0, np.nan).fillna(1)
    df[col]  = df[col] / medians

## 9. Outlier clipping (IQR ×3)

In [ ]:
for col in TIME_COLS:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR    = Q3 - Q1
    df[col] = df[col].clip(lower=Q1 - 3*IQR, upper=Q3 + 3*IQR)

## 10. Key encoding — flags only

In [ ]:
top_keys = df["key"].value_counts().head(TOP_N_KEYS).index.tolist()
for k in top_keys:
    safe = k.replace(" ", "space").replace("/", "slash")
    df[f"key_is_{safe}"] = (df["key"] == k).astype(np.int8)

df["is_vowel"]    = df["key"].apply(lambda k: int(k.lower() in VOWELS))
df["is_special"]  = df["key"].apply(lambda k: int(k.lower() in SPECIAL_KEYS))
df["is_backspace"]= (df["key"] == "backspace").astype(np.int8)
df["is_space"]    = (df["key"] == "space").astype(np.int8)
df["is_digit"]    = df["key"].apply(lambda k: int(k.isdigit()))

all_users       = df["user_id"].unique()
train_user_ids, _ = train_test_split(all_users, test_size=TEST_SIZE,
                                     random_state=RANDOM_STATE)
train_user_ids  = set(train_user_ids)

df["prev_key"] = df.groupby("user_id")["key"].shift(1).fillna("")
df["bigram"]   = df["prev_key"] + "_" + df["key"]

train_bigram_counts = (df[df["user_id"].isin(train_user_ids)]["bigram"]
                       .value_counts())
top_bigrams = train_bigram_counts.head(10).index.tolist()
print(f" [FIX-1] Top bigrams selected from {len(train_user_ids)} train users only")
print(f"         Top bigrams: {top_bigrams[:5]} ...")

for bg in top_bigrams:
    safe = bg.replace(" ", "space").replace("/", "slash")
    df[f"bg_{safe}"] = (df["bigram"] == bg).astype(np.int8)

df.drop(columns=["key", "prev_key", "bigram"], inplace=True)

key_cols    = [c for c in df.columns if c.startswith("key_is_")]
bigram_cols = [c for c in df.columns if c.startswith("bg_")]
print(f" Key cols   : {len(key_cols)}")
print(f" Bigram cols: {len(bigram_cols)}")
print(f" Shape      : {df.shape}")

 [FIX-1] Top bigrams selected from 92 train users only
         Top bigrams: ['space_t', 'e_space', 's_space', 't_h', 'space_i'] ...
 Key cols   : 20
 Bigram cols: 10
 Shape      : (192849, 40)


## 11. Sliding window feature extraction

In [ ]:
def extract_features(w, user_id):
    f = {"user_id": user_id}
    for col in TIME_COLS:
        s = w[col]
        f[f"{col}_mean"]   = s.mean()
        f[f"{col}_std"]    = s.std()
        f[f"{col}_median"] = s.median()
        f[f"{col}_min"]    = s.min()
        f[f"{col}_max"]    = s.max()
        f[f"{col}_skew"]   = float(s.skew())
        f[f"{col}_range"]  = s.max() - s.min()
        f[f"{col}_p10"]    = s.quantile(0.10)
        f[f"{col}_p90"]    = s.quantile(0.90)
        f[f"{col}_iqr"]    = s.quantile(0.75) - s.quantile(0.25)
    for col in ["hold_time", "press_to_press"]:
        s = w[col]
        f[f"{col}_cv"] = (s.std() / s.mean()) if s.mean() != 0 else 0.0
    # Autocorrelation lag-1
    for col, key in [("press_to_press", "ptp"), ("hold_time", "ht")]:
        arr = w[col].values
        if len(arr) > 2 and np.std(arr) > 0:
            f[f"{key}_autocorr"] = float(np.corrcoef(arr[:-1], arr[1:])[0, 1])
        else:
            f[f"{key}_autocorr"] = 0.0
    # Rhythm features
    mp = w["press_to_press"].mean()
    mh = w["hold_time"].mean()
    f["hold_flight_ratio"] = (mh / mp) if mp != 0 else 0.0
    f["burst_speed_mean"]  = w["press_to_press"].nsmallest(10).mean()
    f["burst_speed_p10"]   = w["press_to_press"].quantile(0.10)
    slow = w["press_to_press"].quantile(0.75)
    fast = w["press_to_press"].quantile(0.10)
    f["pause_ratio"]       = (w["press_to_press"] > slow).mean()
    f["fast_key_ratio"]    = (w["press_to_press"] <= fast).mean()
    # Cross-feature ratios
    r2p_m = w["release_to_press"].mean()
    f["ht_p2p_ratio"]      = mh / (mp    + 1e-9)
    f["ht_r2p_ratio"]      = mh / (r2p_m + 1e-9)
    f["p2p_r2p_ratio"]     = mp / (r2p_m + 1e-9)
    f["rhythm_stability"]  = w["press_to_press"].std() / (mp + 1e-9)
    # Error behavior
    f["backspace_ratio"]   = w["is_backspace"].mean()
    hd = w["hold_time"].diff().dropna()
    f["hold_diff_mean"]    = hd.mean()
    f["hold_diff_std"]     = hd.std()
    # Key/bigram frequencies
    for col in key_cols:
        f[f"{col}_freq"] = w[col].mean()
    for col in bigram_cols:
        f[f"{col}_freq"] = w[col].mean()
    # Behavioral ratios
    f["vowel_ratio"]   = w["is_vowel"].mean()
    f["special_ratio"] = w["is_special"].mean()
    f["space_ratio"]   = w["is_space"].mean()
    f["digit_ratio"]   = w["is_digit"].mean()
    return f

all_windows  = []
total_users  = df["user_id"].nunique()
for i, (uid, udf) in enumerate(df.groupby("user_id")):
    udf = udf.reset_index(drop=True)
    n   = len(udf)
    for start in range(0, n - WINDOW_SIZE + 1, STEP_SIZE):
        w = udf.iloc[start: start + WINDOW_SIZE]
        all_windows.append(extract_features(w, uid))
    if (i+1) % 20 == 0 or (i+1) == total_users:
        print(f" Processed {i+1}/{total_users} users...")

windows_df = pd.DataFrame(all_windows).fillna(0)
wpu = windows_df.groupby("user_id").size()
print(f"\n Window matrix : {windows_df.shape}")
print(f" Windows/user — min:{wpu.min()} mean:{wpu.mean():.0f} max:{wpu.max()}")

 Processed 20/115 users...
 Processed 40/115 users...
 Processed 60/115 users...
 Processed 80/115 users...
 Processed 100/115 users...
 Processed 115/115 users...

 Window matrix : (7317, 91)
 Windows/user — min:40 mean:64 max:101


## 12. Cap windows per user

In [ ]:
parts = []
for uid, udf in windows_df.groupby("user_id"):
    if len(udf) > CAP_PER_USER:
        udf = udf.sample(n=CAP_PER_USER, random_state=RANDOM_STATE)
    parts.append(udf)
windows_bal = pd.concat(parts, ignore_index=True)

## 13. Zero-variance removal

In [ ]:
FEAT_COLS = [c for c in windows_bal.columns if c != "user_id"]
X_raw     = windows_bal[FEAT_COLS].values
vt        = VarianceThreshold(threshold=0.0)
X_vt      = vt.fit_transform(X_raw)
kept      = np.array(FEAT_COLS)[vt.get_support()].tolist()

## 14. Correlation pruning

In [ ]:
feat_df  = pd.DataFrame(X_vt, columns=kept)
corr_mat = feat_df.corr().abs()
upper    = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
to_drop  = [c for c in upper.columns if any(upper[c] > CORR_THRESH)]
feat_df  = feat_df.drop(columns=to_drop)
FEAT_FINAL = feat_df.columns.tolist()
print(f" Dropped {len(to_drop)} correlated features | Final: {len(FEAT_FINAL)}")

X_all = feat_df.values
y_all = windows_bal["user_id"].values

 Dropped 6 correlated features | Final: 84


## 15. Split First

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE,
    random_state=RANDOM_STATE, stratify=y_all)

## 16. Scale after split (fit on train only)

In [ ]:
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"\n Train: {X_train_sc.shape}  Test: {X_test_sc.shape}")
print(f" Users: {len(np.unique(y_train))}")


 Train: (5853, 84)  Test: (1464, 84)
 Users: 115


## 17. Save

In [ ]:
np.save("outputs/X_train.npy", X_train_sc)
np.save("outputs/X_test.npy",  X_test_sc)
np.save("outputs/y_train.npy", y_train)
np.save("outputs/y_test.npy",  y_test)
joblib.dump(scaler,"outputs/scaler.pkl")
joblib.dump(FEAT_FINAL,"outputs/feature_cols.pkl")
joblib.dump(vt,"outputs/variance_threshold.pkl")

print(BANNER)
print("PREPROCESSING COMPLETE")
print(BANNER)

PREPROCESSING COMPLETE


# MULTICLASS CLASSIFICATION

## 18. Load

In [ ]:
le           = LabelEncoder()
y_train_enc  = le.fit_transform(y_train)
y_test_enc   = le.transform(y_test)
n_classes    = len(le.classes_)
print(f" Classes: {n_classes}")

 Classes: 115


## 19. Result Tracker

In [ ]:
all_results = []
all_preds   = {}
all_probas  = {}

def evaluate(name, y_true, y_pred, y_proba=None, t_elapsed=None):
    acc  = accuracy_score(y_true, y_pred) * 100
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0) * 100
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0) * 100
    f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0) * 100
    top5 = None
    if y_proba is not None:
        try:
            top5 = top_k_accuracy_score(y_true, y_proba,
                                        k=min(5, n_classes),
                                        labels=np.unique(y_train)) * 100
        except Exception:
            pass
    all_results.append({
        'Model':       name,
        'Accuracy (%)':  round(acc,  2),
        'Precision (%)': round(prec, 2),
        'Recall (%)':    round(rec,  2),
        'F1 Score (%)':  round(f1,   2),
        'Top-5 Acc (%)': round(top5, 2) if top5 is not None else '-',
        'Time (s)':      round(t_elapsed, 1) if t_elapsed else '-',
    })
    all_preds[name]  = y_pred
    all_probas[name] = y_proba
    print(f" Accuracy : {acc:.2f}%  Precision: {prec:.2f}%  "
          f"Recall: {rec:.2f}%  F1: {f1:.2f}%", end="")
    if top5: print(f"  Top-5: {top5:.2f}%", end="")
    if t_elapsed: print(f"  [{t_elapsed:.1f}s]", end="")
    print()

## 20. Logistic Regression

In [ ]:
print("\n--- MODEL 1: Logistic Regression ---")
t0  = time.time()
lr  = LogisticRegression(max_iter=5000, C=1.0, random_state=42, n_jobs=-1)
lr.fit(X_train_sc, y_train_enc)
t   = time.time() - t0
evaluate("Logistic Regression", y_test,
         le.inverse_transform(lr.predict(X_test_sc)),
         lr.predict_proba(X_test_sc), t)


--- MODEL 1: Logistic Regression ---
 Accuracy : 88.46%  Precision: 89.26%  Recall: 88.46%  F1: 88.39%  Top-5: 99.11%  [6.0s]


## 21. Naive Bayes

In [ ]:
print("\n--- MODEL 2: Naive Bayes ---")
t0  = time.time()
gnb = GaussianNB()
gnb.fit(X_train_sc, y_train_enc)
t   = time.time() - t0
evaluate("Naive Bayes", y_test,
         le.inverse_transform(gnb.predict(X_test_sc)),
         gnb.predict_proba(X_test_sc), t)


--- MODEL 2: Naive Bayes ---
 Accuracy : 73.16%  Precision: 78.33%  Recall: 73.16%  F1: 72.83%  Top-5: 94.81%  [0.0s]


## 22. KNN

In [ ]:
print("\n--- MODEL 3: KNN (k=7) ---")
t0  = time.time()
knn = KNeighborsClassifier(n_neighbors=7, n_jobs=-1)
knn.fit(X_train_sc, y_train_enc)
t   = time.time() - t0
evaluate("KNN (k=7)", y_test,
         le.inverse_transform(knn.predict(X_test_sc)),
         knn.predict_proba(X_test_sc), t)


--- MODEL 3: KNN (k=7) ---
 Accuracy : 62.36%  Precision: 67.10%  Recall: 62.36%  F1: 62.11%  Top-5: 85.52%  [0.0s]


## 23. SVM RBF

In [ ]:
print("\n--- MODEL 4: SVM RBF ---")
t0  = time.time()
svm = SVC(kernel='rbf', C=10, probability=True, random_state=42)
svm.fit(X_train_sc, y_train_enc)
t   = time.time() - t0
evaluate("SVM (C=10, RBF)", y_test,
         le.inverse_transform(svm.predict(X_test_sc)),
         svm.predict_proba(X_test_sc), t)


--- MODEL 4: SVM RBF ---
 Accuracy : 92.49%  Precision: 93.09%  Recall: 92.49%  F1: 92.42%  Top-5: 98.84%  [29.1s]


## 24. Random Forest  

In [ ]:
print("\n--- MODEL 5: Random Forest ---")
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=300, max_depth=20, min_samples_leaf=3,
    max_features='sqrt', class_weight='balanced',
    oob_score=True, random_state=42, n_jobs=-1)
rf.fit(X_train_sc, y_train)
t  = time.time() - t0
print(f" OOB Score: {rf.oob_score_*100:.2f}%")
evaluate("Random Forest", y_test,
         rf.predict(X_test_sc),
         rf.predict_proba(X_test_sc), t)
joblib.dump(rf, 'outputs/model_rf_v7.pkl')


--- MODEL 5: Random Forest ---
 OOB Score: 96.26%
 Accuracy : 96.72%  Precision: 96.96%  Recall: 96.72%  F1: 96.68%  Top-5: 99.80%  [5.7s]


['outputs/model_rf_v7.pkl']

## 25. XGBoost

In [ ]:
print("\n--- MODEL 6: XGBoost ---")
t0  = time.time()
xgb = XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.08,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    eval_metric='mlogloss', random_state=42, n_jobs=-1, verbosity=0)
xgb.fit(X_train_sc, y_train_enc)
t   = time.time() - t0
evaluate("XGBoost", y_test,
         le.inverse_transform(xgb.predict(X_test_sc)),
         xgb.predict_proba(X_test_sc), t)
joblib.dump(xgb, 'outputs/model_xgb_v7.pkl')


--- MODEL 6: XGBoost ---
 Accuracy : 97.06%  Precision: 97.23%  Recall: 97.06%  F1: 97.02%  Top-5: 99.86%  [66.6s]


['outputs/model_xgb_v7.pkl']

## 26. LightGBM

In [ ]:
print("\n--- MODEL 7: LightGBM ---")
t0  = time.time()
lgb = LGBMClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    num_leaves=63, subsample=0.8, colsample_bytree=0.8,
    class_weight='balanced', random_state=42, verbose=-1)
lgb.fit(X_train_sc, y_train_enc)
t   = time.time() - t0
evaluate("LightGBM", y_test,
         le.inverse_transform(lgb.predict(X_test_sc)),
         lgb.predict_proba(X_test_sc), t)
joblib.dump(lgb, 'outputs/model_lgb_v7.pkl')


--- MODEL 7: LightGBM ---
 Accuracy : 97.88%  Precision: 98.04%  Recall: 97.88%  F1: 97.86%  Top-5: 99.86%  [80.4s]


['outputs/model_lgb_v7.pkl']

## 27. Extra Trees

In [ ]:
print("\n--- MODEL 8: Extra Trees ---")
t0 = time.time()
et = ExtraTreesClassifier(
    n_estimators=300, class_weight='balanced',
    n_jobs=-1, random_state=42)
et.fit(X_train_sc, y_train)
t  = time.time() - t0
evaluate("Extra Trees", y_test,
         et.predict(X_test_sc),
         et.predict_proba(X_test_sc), t)


--- MODEL 8: Extra Trees ---
 Accuracy : 97.95%  Precision: 98.11%  Recall: 97.95%  F1: 97.95%  Top-5: 100.00%  [2.3s]


## 28. MLP Neural Net

In [ ]:
print("\n--- MODEL 9: MLP Neural Net ---")
t0  = time.time()
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32), activation='relu',
    max_iter=500, early_stopping=True,
    validation_fraction=0.1, random_state=42)
mlp.fit(X_train_sc, y_train_enc)
t   = time.time() - t0
evaluate("MLP Neural Net", y_test,
         le.inverse_transform(mlp.predict(X_test_sc)),
         mlp.predict_proba(X_test_sc), t)


--- MODEL 9: MLP Neural Net ---
 Accuracy : 90.37%  Precision: 91.19%  Recall: 90.37%  F1: 90.34%  Top-5: 99.45%  [20.8s]


## 29. Results Table

In [ ]:
print(f"\n{'='*85}")
print("RESULTS SUMMARY")
print(f"{'='*85}")
results_df = (pd.DataFrame(all_results)
              .sort_values("Accuracy (%)", ascending=False)
              .reset_index(drop=True))

print(f"\n {'Model':<26} {'Accuracy':>10} {'F1':>8} {'Top-5':>8} {'Time(s)':>8}")
print(f" {'─'*26} {'─'*10} {'─'*8} {'─'*8} {'─'*8}")
for i, r in results_df.iterrows():
    marker  = " ◄ best" if i == 0 else ""
    top5str = f"{r['Top-5 Acc (%)']:>8.2f}" if r['Top-5 Acc (%)'] != '-' else "       -"
    print(f" {r['Model']:<26} {r['Accuracy (%)']:>10.2f} "
          f"{r['F1 Score (%)']:>8.2f} {top5str} {str(r['Time (s)']):>8}{marker}")

results_df.to_csv("outputs/model_comparison_v7.csv", index=False)
joblib.dump(le, 'outputs/label_encoder_v7.pkl')


RESULTS SUMMARY

 Model                        Accuracy       F1    Top-5  Time(s)
 ────────────────────────── ────────── ──────── ──────── ────────
 Extra Trees                     97.95    97.95   100.00      2.3 ◄ best
 LightGBM                        97.88    97.86    99.86     80.4
 XGBoost                         97.06    97.02    99.86     66.6
 Random Forest                   96.72    96.68    99.80      5.7
 SVM (C=10, RBF)                 92.49    92.42    98.84     29.1
 MLP Neural Net                  90.37    90.34    99.45     20.8
 Logistic Regression             88.46    88.39    99.11      6.0
 Naive Bayes                     73.16    72.83    94.81      0.0
 KNN (k=7)                       62.36    62.11    85.52      0.0


['outputs/label_encoder_v7.pkl']